# FastPower (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_python.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `FastPower`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno i nomi dello pseudocodice. Due eccezioni in Python, dove `from` e `next` hanno un significato speciale: `Hanoi` usa `frm` al posto di `from` e `IterativeFib` usa `nxt` al posto di `next`.

## Strumenti: contatori, stack e traccia

La cella definisce i contatori (`cnt`, `calls`, `max_depth`), lo stack delle chiamate (`push` e `pop`) e le funzioni che stampano la traccia (`event`, `say`) e la tabella dei passi (`print_steps`). Basta eseguirla una volta.

In [1]:
cnt = [0] * 20      # cnt[r] = numero di esecuzioni della riga r (righe numerate come nelle dispense)
calls = 0           # numero di chiamate = nodi dell'albero delle chiamate
max_depth = 0       # profondita' della ricorsione = altezza massima dello stack
mults = 0           # moltiplicazioni (FastPower)
moves = 0           # mosse (Hanoi)
ev = 0              # contatore degli eventi della traccia
stack = []          # lo stack delle chiamate: gli argomenti delle chiamate attive
seen = [0] * 64     # seen[x] = numero di chiamate con argomento x
trace = False       # True: stampa la traccia dell'esecuzione


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # inizia una chiamata: il suo frame va in cima allo stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # la chiamata termina: il suo frame esce dallo stack
    stack.pop()


def event(text):                   # riga numerata della traccia, con lo stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Riga | Esecuzioni")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Passi totali S = %d" % sum(cnt))

## Elevamento a potenza veloce: FastPower

**Input:** un numero $x$ e un numero naturale $n$. **Output:** $x^n$, per quadrati ripetuti. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor + 2$ chiamate, al più $2(\lfloor\log_2 n\rfloor + 1)$ moltiplicazioni.

Pseudocodice: sezione 14.7 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
def floor_log2(n):                                 # floor(log2 n) con un ciclo (solo per la formula dei passi)
    k = 0
    while n > 1:
        n = n // 2
        k = k + 1
    return k


def fast_power(x, n):
    global mults
    push(n); cnt[2] += 1
    if n == 0:
        cnt[3] += 1
        if trace: say("FastPower(%d, 0)" % x, "caso base, restituisce 1")
        pop()
        return 1
    if trace: say("FastPower(%d, %d)" % (x, n), "%d != 0 => chiama FastPower(%d, %d)   (%d/2 = %d)" % (n, x, n // 2, n, n // 2))
    y = fast_power(x, n // 2); cnt[5] += 1         # una sola chiamata ricorsiva: il risultato e' in y
    cnt[6] += 1
    if n % 2 == 0:
        cnt[7] += 1
        mults += 1
        r = y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d e' pari => restituisce %d * %d = %d" % (y, n, y, y, r))
    else:
        cnt[9] += 1
        mults += 2
        r = x * y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d e' dispari => restituisce %d * %d * %d = %d" % (y, n, x, y, y, r))
    pop()
    return r

In [3]:
# Esempio delle dispense: FastPower(3, 13)
x, n = 3, 13
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, chiamate = %d, moltiplicazioni = %d" % (x, n, r, calls, mults))
print_steps(2, 3, 5, 6, 7, 9)
print("S(n) = 4 floor(log2 n) + 6 = %d" % (4 * floor_log2(n) + 6))

# Quante moltiplicazioni: circa 2 log2 n invece di n (x = 1, per non traboccare)
print()
print("      n | chiamate | moltiplicazioni | S(n) | 4 floor(log2 n) + 6")
for n in [1, 2, 13, 100, 1000, 1000000]:
    reset()
    fast_power(1, n)
    print("%7d | %8d | %15d | %4d | %d" % (n, calls, mults, sum(cnt), 4 * floor_log2(n) + 6))

chiamata                    | azione
FastPower(3, 13)            | 13 != 0 => chiama FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => chiama FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => chiama FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => chiama FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | caso base, restituisce 1
      FastPower(3, 1)       | y = 1, 1 e' dispari => restituisce 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 e' dispari => restituisce 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 e' pari => restituisce 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 e' dispari => restituisce 3 * 729 * 729 = 1594323
3^13 = 1594323, chiamate = 5, moltiplicazioni = 7
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Passi totali S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n | chiamate | moltiplicazioni | S(n) | 4 f

**Prova tu.** Cambia $x$ e $n$ e conta le moltiplicazioni.

In [4]:
x, n = 2, 10        # <-- cambia x e n (con x e n grandi C++ e Java traboccano)
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, moltiplicazioni = %d" % (x, n, r, mults))

chiamata                    | azione
FastPower(2, 10)            | 10 != 0 => chiama FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => chiama FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => chiama FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => chiama FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | caso base, restituisce 1
      FastPower(2, 1)       | y = 1, 1 e' dispari => restituisce 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 e' pari => restituisce 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 e' dispari => restituisce 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 e' pari => restituisce 32 * 32 = 1024
2^10 = 1024, moltiplicazioni = 6


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).